In [8]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [9]:
import pandas as pd

df = pd.read_csv("../data/customer_purchase_data.csv")

df.head()
df.info()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   age           1000 non-null   int64  
 1   income        1000 non-null   int64  
 2   gender        1000 non-null   object 
 3   city          1000 non-null   object 
 4   time_on_site  1000 non-null   float64
 5   pages_viewed  1000 non-null   int64  
 6   purchased     1000 non-null   int64  
dtypes: float64(1), int64(4), object(2)
memory usage: 54.8+ KB


,age,income,time_on_site,pages_viewed,purchased
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,38.745000,69785.694000,15.483800,10.371000,0.385000
std,12.186734,28440.126145,8.363308,5.310891,0.486839
min,18.000000,20060.000000,1.000000,1.000000,0.000000
25%,28.000000,46114.500000,8.300000,6.000000,0.000000
50%,40.000000,69210.500000,15.600000,11.000000,0.000000
75%,50.000000,95215.500000,22.700000,15.000000,1.000000
max,59.000000,119986.000000,30.000000,19.000000,1.000000


In [10]:
df["purchased"].value_counts(normalize=True)


purchased
0    0.615
1    0.385
Name: proportion, dtype: float64

In [11]:
from sklearn.model_selection import train_test_split

X = df.drop("purchased", axis=1)
y = df["purchased"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train.shape, X_test.shape


((800, 6), (200, 6))

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report

num_cols = ["age", "income", "time_on_site", "pages_viewed"]
cat_cols = ["gender", "city"]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

log_reg_pipeline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000))
])

log_reg_pipeline.fit(X_train, y_train)
y_pred_lr = log_reg_pipeline.predict(X_test)

print("Logistic Regression Results")
print(classification_report(y_test, y_pred_lr))


Logistic Regression Results
              precision    recall  f1-score   support

           0       0.73      0.78      0.75       123
           1       0.60      0.53      0.57        77

    accuracy                           0.69       200
   macro avg       0.67      0.66      0.66       200
weighted avg       0.68      0.69      0.68       200



In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline

rf_pipeline = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(random_state=42))
])

param_dist = {
    "clf__n_estimators": [200, 300, 500],
    "clf__max_depth": [6, 10, 14, None],
    "clf__min_samples_leaf": [2, 5, 10]
}

search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

best_model = search.best_estimator_

In [13]:
from sklearn.metrics import classification_report

y_pred_best = best_model.predict(X_test)
print("Tuned Random Forest Results")
print(classification_report(y_test, y_pred_best))


Tuned Random Forest Results
              precision    recall  f1-score   support

           0       0.84      0.89      0.87       123
           1       0.81      0.73      0.77        77

    accuracy                           0.83       200
   macro avg       0.83      0.81      0.82       200
weighted avg       0.83      0.83      0.83       200

